# 2. Подготовка тензоров, Full FT и LoRA

Обучение на GPU-сервере с загрузкой токенизатора и модели. Функции подготовки данных и обучения находятся в `data.py` и `training.py`.

Каждый опыт выполняется отдельно с собственными именем и конфигурацией. Перезапуск kernel между опытами освобождает предыдущую модель, Trainer и optimizer.

In [1]:
%load_ext autoreload
%autoreload 2
import json
import hashlib
from pathlib import Path
from functools import partial
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, set_seed
from data import make_prompt, make_target, tokenize_example
from training import (
    MODEL_NAME, MODEL_REVISION, load_base_model, attach_lora,
    count_parameters, train_model, make_lora_experiments,
)

ROOT = Path.cwd()
assert (ROOT / 'training.py').is_file(), 'Рабочая папка notebook должна быть hw4_lora'
assert torch.cuda.is_available(), 'Для обучения и измерения GPU-памяти требуется CUDA'
print(torch.cuda.get_device_name(0))
SEED = 42
set_seed(SEED)
DEVICE = 'cuda'
USE_BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print('dtype:', DTYPE)

NVIDIA A100-SXM4-80GB
dtype: torch.bfloat16


## Одинаковые исходные данные

Notebook читает готовые JSONL без вызова загрузчика. Сравнение использует одинаковые данные и сопоставимые число шагов и критерий выбора checkpoint. Test в обучении не участвует.

In [2]:
raw = DatasetDict({
    split: Dataset.from_list([
        json.loads(line) for line in (ROOT / 'data/wikiwhy' / f'{split}.jsonl').read_text().splitlines()
    ])
    for split in ('train', 'validation')
})
print(raw)
MAX_LENGTH = 1024  # Максимальная длина входа с ответом; более длинные последовательности усекаются.
SCOPE = 'all_topics'
# Один состав данных для обоих режимов обучения.

DatasetDict({
    train: Dataset({
        features: ['id', 'ctx', 'title', 'topic', 'split', 'question', 'cause', 'effect', 'explanation'],
        num_rows: 7397
    })
    validation: Dataset({
        features: ['id', 'ctx', 'title', 'topic', 'split', 'question', 'cause', 'effect', 'explanation'],
        num_rows: 924
    })
})


## Токенизация одного примера

`tokenize_example` в `data.py` возвращает списки длины `[L]`. Значение `labels=-100` исключает позиции из loss. Causal LM сдвигает цели внутри `forward`.

`apply_chat_template` учитывает специальные токены Qwen. Обучающий вход содержит полный диалог с ответом assistant; вход для генерации содержит вопрос и начало ответа. Маска исключает prompt из loss.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
tokenizer.padding_side = 'right'
assert tokenizer.pad_token_id is not None, 'У токенизатора не задан pad_token_id'
print('pad/eos:', tokenizer.pad_token_id, tokenizer.eos_token_id)

example = raw['train'][0]
encoded = tokenize_example(example, tokenizer, max_length=MAX_LENGTH)
assert set(('input_ids', 'attention_mask', 'labels')) <= set(encoded)
assert len(encoded['input_ids']) == len(encoded['attention_mask']) == len(encoded['labels'])
assert 0 < len(encoded['input_ids']) <= MAX_LENGTH
assert any(label != -100 for label in encoded['labels']), 'После усечения ответ не должен исчезать целиком'
print(tokenizer.decode(encoded['input_ids']))
print('Первые пары input_id / label:', list(zip(encoded['input_ids'], encoded['labels']))[:30])
print('Обучаемых позиций:', sum(label != -100 for label in encoded['labels']))

pad/eos: 151643 151645
<|im_start|>system
You are a helpful assistant. Answer the question using the requested format: Answer: ... followed by Explanation: ...<|im_end|>
<|im_start|>user
Why did Harry tells Ros that he was looking for Adam's mother, but could not find her in the episode of Spooks titled "Split Loyalties"?<|im_end|>
<|im_start|>assistant
Answer: Because, before his death, Adam altered her records on where she lived.
Explanation: People often find people by looking up where there records state they are supposed to be. When records are altered, it is more difficult to find people.<|im_end|>

Первые пары input_id / label: [(151644, -100), (8948, -100), (198, -100), (2610, -100), (525, -100), (264, -100), (10950, -100), (17847, -100), (13, -100), (21806, -100), (279, -100), (3405, -100), (1667, -100), (279, -100), (11223, -100), (3561, -100), (25, -100), (21806, -100), (25, -100), (2503, -100), (8110, -100), (553, -100), (71287, -100), (25, -100), (2503, -100), (151645, -10

In [4]:
prepared = raw.map(
    partial(tokenize_example, tokenizer=tokenizer, max_length=MAX_LENGTH),
    remove_columns=raw['train'].column_names,
    load_from_cache_file=False,
)
# Collator дополняет готовые labels, сохраняя маску prompt.
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=None, padding=True, label_pad_token_id=-100)
batch = collator([prepared['train'][0], prepared['train'][1]])
print({key: tuple(value.shape) for key, value in batch.items()})
assert batch['input_ids'].shape == batch['attention_mask'].shape == batch['labels'].shape
assert torch.all(batch['labels'][batch['attention_mask'] == 0] == -100)

Map:   0%|          | 0/7397 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

{'input_ids': (2, 117), 'attention_mask': (2, 117), 'labels': (2, 117)}


## Конфиг одного опыта

Стартовый конфиг задаёт условия сравнения Full FT и LoRA. `gradient_checkpointing_kwargs` с `use_reentrant=False` поддерживает обучение LoRA при замороженных исходных весах.

Для Full FT используется `ADAPTER_CONFIG=None`, для LoRA baseline — `ADAPTER_CONFIG=LORA_BASELINE`. `RUN_NAME` определяет отдельный каталог опыта. Learning rate может подбираться на validation для сравнения методов. В исследованиях четырёх параметров LoRA остальные настройки обучения фиксированы. Full FT выполняется без квантования.

In [5]:
RUN_NAME = 'full_ft'
LORA_BASELINE = {
    'r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'v_proj'],
    'task_type': 'CAUSAL_LM',
    'bias': 'none',
}
ADAPTER_CONFIG = None
TRAINING_CONFIG = {
    'learning_rate': 2e-5,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 16,
    'gradient_checkpointing': True,
    'gradient_checkpointing_kwargs': {'use_reentrant': False},
    'bf16': USE_BF16,
    'fp16': not USE_BF16,
    'eval_strategy': 'epoch',
    'save_strategy': 'epoch',
    'save_total_limit': 1,
    'logging_steps': 10,
    'report_to': 'none',
    'seed': SEED,
    'data_seed': SEED,
}
OUTPUT_DIR = ROOT / 'runs' / RUN_NAME
assert not OUTPUT_DIR.exists(), 'Каталог опыта уже существует; перезапись запрещена'
print(OUTPUT_DIR)

/home/edu/amsafin/HSE-Avito/LLM/hw4_lora/runs/full_ft


## Модель и обучаемые параметры

`load_base_model` загружает исходные веса закреплённой ревизии. Для LoRA PEFT добавляет адаптеры; Full FT использует исходную модель без адаптеров.

`model.named_modules()` возвращает имена слоёв. `numel()` и `requires_grad` определяют общее число параметров и число обучаемых параметров.

In [6]:
model = load_base_model(MODEL_NAME, revision=MODEL_REVISION, dtype=DTYPE, device=DEVICE)
print('Линейные слои:', [name for name, module in model.named_modules() if isinstance(module, torch.nn.Linear)][:20])
if ADAPTER_CONFIG is not None:
    model = attach_lora(model, ADAPTER_CONFIG)
print(count_parameters(model))

Линейные слои: ['model.layers.0.self_attn.q_proj', 'model.layers.0.self_attn.k_proj', 'model.layers.0.self_attn.v_proj', 'model.layers.0.self_attn.o_proj', 'model.layers.0.mlp.gate_proj', 'model.layers.0.mlp.up_proj', 'model.layers.0.mlp.down_proj', 'model.layers.1.self_attn.q_proj', 'model.layers.1.self_attn.k_proj', 'model.layers.1.self_attn.v_proj', 'model.layers.1.self_attn.o_proj', 'model.layers.1.mlp.gate_proj', 'model.layers.1.mlp.up_proj', 'model.layers.1.mlp.down_proj', 'model.layers.2.self_attn.q_proj', 'model.layers.2.self_attn.k_proj', 'model.layers.2.self_attn.v_proj', 'model.layers.2.self_attn.o_proj', 'model.layers.2.mlp.gate_proj', 'model.layers.2.mlp.up_proj']
{'trainable_parameters': 1543714304, 'total_parameters': 1543714304}


## Обучение и измерения

`train_model` создаёт Trainer, запускает обучение и сохраняет модель или адаптер и токенизатор прямо в `OUTPUT_DIR`.

Возвращаемые поля: `train_seconds`, `peak_gpu_memory_gb`, `trainable_parameters`, `total_parameters`. Пик выделенной памяти PyTorch пересчитывается в GiB как байты / 1024³. Время охватывает `train()`, включая промежуточную валидацию и сохранение checkpoint внутри него. Загрузка модели и итоговое сохранение после `train()` в замер не входят.

In [7]:
config_record = {
    'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION,
    'max_length': MAX_LENGTH, 'scope': SCOPE, 'seed': SEED,
    'training': TRAINING_CONFIG, 'lora': ADAPTER_CONFIG,
    'data_manifest_sha256': hashlib.sha256((ROOT / 'data/manifest.json').read_bytes()).hexdigest(),
}
# train_model создаёт OUTPUT_DIR и сохраняет туда модель/адаптер.
train_stats = train_model(
    model, tokenizer, prepared['train'], prepared['validation'],
    collator, TRAINING_CONFIG, OUTPUT_DIR,
)
assert {'train_seconds', 'peak_gpu_memory_gb', 'trainable_parameters', 'total_parameters'} <= set(train_stats)
with (OUTPUT_DIR / 'run_config.json').open('x') as file:
    json.dump(config_record, file, ensure_ascii=False, indent=2)
with (OUTPUT_DIR / 'train_stats.json').open('x') as file:
    json.dump(train_stats, file, ensure_ascii=False, indent=2)
print(train_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,1.040600,2.034235


/home/edu/amsafin/HSE-Avito/LLM/hw4_lora/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'train_seconds': 638.2669136610348, 'peak_gpu_memory_gb': 15.273505210876465, 'trainable_parameters': 1543714304, 'total_parameters': 1543714304}


## Четыре исследования LoRA

`make_lora_experiments` возвращает baseline и варианты с отдельным изменением `r`, `lora_alpha`, `lora_dropout` или `target_modules`. Каждый вариант оценивается по обеим метрикам из notebook `03`.

План конфигураций не запускает обучение автоматически. `ADAPTER_CONFIG` задаёт выбранный вариант, `RUN_NAME` — имя опыта. Каждый опыт начинается с новой исходной модели после перезапуска kernel.

При фиксированном alpha изменение r также меняет масштаб alpha/r. Этот эффект учитывается при интерпретации результатов.

In [8]:
lora_experiments = make_lora_experiments(LORA_BASELINE)
print(lora_experiments)

{'baseline': {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'bias': 'none'}, 'r_8': {'r': 8, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'bias': 'none'}, 'r_32': {'r': 32, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'bias': 'none'}, 'alpha_16': {'r': 16, 'lora_alpha': 16, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'bias': 'none'}, 'alpha_64': {'r': 16, 'lora_alpha': 64, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'bias': 'none'}, 'dp_0.0': {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.0, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'bias': 'none'}, 'dp_0.1': {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.1, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'bias': 'none'}, 

In [10]:
import gc

LORA_BASELINE = {
    'r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'v_proj'],
    'task_type': 'CAUSAL_LM',
    'bias': 'none',
}
TRAINING_CONFIG = {
    'learning_rate': 2e-5,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 32,
    'per_device_eval_batch_size': 32,
    'gradient_accumulation_steps': 1,
    'gradient_checkpointing': True,
    'gradient_checkpointing_kwargs': {'use_reentrant': False},
    'bf16': USE_BF16,
    'fp16': not USE_BF16,
    'eval_strategy': 'epoch',
    'save_strategy': 'epoch',
    'save_total_limit': 1,
    'logging_steps': 10,
    'report_to': 'none',
    'seed': SEED,
    'data_seed': SEED,
}

for key, adapter_config in make_lora_experiments(LORA_BASELINE).items():
    run_name = 'lora_baseline' if key == 'baseline' else f'lora_{key}'
    output_dir = ROOT / 'runs' / run_name
    if (output_dir / 'train_stats.json').is_file():
        print('уже есть', run_name)
        continue
    model = load_base_model(MODEL_NAME, revision=MODEL_REVISION, dtype=DTYPE, device=DEVICE)
    model = attach_lora(model, adapter_config)
    model.enable_input_require_grads()
    train_stats = train_model(
        model, tokenizer, prepared['train'], prepared['validation'],
        collator, TRAINING_CONFIG, output_dir,
    )
    config_record = {
        'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION,
        'max_length': MAX_LENGTH, 'scope': SCOPE, 'seed': SEED,
        'training': TRAINING_CONFIG, 'lora': adapter_config,
        'data_manifest_sha256': hashlib.sha256((ROOT / 'data/manifest.json').read_bytes()).hexdigest(),
    }
    with (output_dir / 'run_config.json').open('x') as file:
        json.dump(config_record, file, ensure_ascii=False, indent=2)
    with (output_dir / 'train_stats.json').open('x') as file:
        json.dump(train_stats, file, ensure_ascii=False, indent=2)
    print(run_name, train_stats)
    del model
    gc.collect()
    torch.cuda.empty_cache()

уже есть lora_baseline


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitu

Epoch,Training Loss,Validation Loss
1,1.130900,2.219782


/home/edu/amsafin/HSE-Avito/LLM/hw4_lora/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


lora_r_8 {'train_seconds': 52.75063813687302, 'peak_gpu_memory_gb': 21.278529167175293, 'trainable_parameters': 1089536, 'total_parameters': 1544803840}


... (26 outputs left out)

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magni

Epoch,Training Loss,Validation Loss
1,1.098200,2.151905


/home/edu/amsafin/HSE-Avito/LLM/hw4_lora/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


lora_qkvo {'train_seconds': 60.417782668955624, 'peak_gpu_memory_gb': 21.315112113952637, 'trainable_parameters': 4358144, 'total_parameters': 1548072448}


## После обучения

Конфигурация и измерения сохраняются в `run_config.json` и `train_stats.json`. Оценка и анализ результатов находятся в `03_evaluation_report.ipynb`.